# CatBoost GPU с динамикой статусов

Один CatBoost учится на пяти утренних стартах прогнозировать долю маршрута относительно сезонного профиля. Дополнительные признаки измеряют изменение статусов за полчаса, час и два часа, отклонение каждого статуса от типичного уровня и изменение по всей сети. Проверяем на двух предтестовых неделях веса 0%, 10% и 25%, сохраняя объём Session6. На локальном test требуем выигрыш от 0.0003 по дневной метрике без ухудшения общего score. После допуска переобучаем новый эксперт на всей доступной истории. Локальная оценка получена до refit, этот test уже изучался.


In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from catboost import CatBoostRegressor
from IPython.display import display

ROOT = Path(os.environ.get('BLEND_PACKAGE', '/root/experiments/session6_blend_server'))
OOF_PATH = None
TEST_PATH = None
OUT = ROOT / 'output_pulse_cat_gpu'
OUT.mkdir(parents=True, exist_ok=True)

BASES = ['session6', 'rf', 'cat_rich', 'xgb_rich', 'xgb_clean', 'profile28']
GROUP_KEYS = ['cutoff', 'horizon']
KEYS = GROUP_KEYS + ['route_id']
COLUMNS = KEYS + BASES + ['y', 'level24', 'current', 'delta1h', 'delta2h', 'network_current']
THREADS = 2
SEED = 42

def read_panel(path, name, start, end):
    archived = ROOT / 'blend_inputs' / 'blend_best_output' / name
    if path is None and archived.is_file():
        path = archived
    if path is None:
        candidates = [p for p in ROOT.rglob(name) if OUT not in p.parents
                      and set(COLUMNS).issubset(pq.ParquetFile(p).schema.names)]
        if len(candidates) != 1:
            raise ValueError(f'Укажите путь явно для {name}: {candidates}')
        path = candidates[0]
    frame = pd.read_parquet(path, columns=COLUMNS)
    frame['cutoff'] = pd.to_datetime(frame.cutoff)
    expected = pd.date_range(start, end, inclusive='left') + pd.Timedelta(hours=10, minutes=30)
    actual = pd.DatetimeIndex(frame.cutoff.unique()).sort_values()
    counts = frame.groupby(GROUP_KEYS).size()
    if (not actual.equals(expected) or frame.duplicated(KEYS).any()
            or set(frame.horizon.unique()) != set(range(1, 9))
            or len(counts) != len(expected)*8 or not counts.eq(1000).all()
            or not np.isfinite(frame[COLUMNS[3:]].to_numpy()).all()
            or frame.y.lt(0).any() or frame.session6.lt(0).any()):
        raise ValueError(f'Некорректная панель: {path}')
    print(path, len(frame))
    return frame.reset_index(drop=True)



DATA = ROOT / 'blend_inputs/session6'
PARENT = Path(os.environ.get('PULSE_CAT_PARENT', str(ROOT/'pulse_cat_parent.csv')))
SIGNALS = ['target_1h'] + [f'status_{i}' for i in range(1,7)]
STEP = pd.Timedelta(minutes=30)
START_MINUTES = (570,600,630,660,690)
VOLUME_SCALE = 1000.


In [ ]:
def with_volume(frame, values, anchor=None):
    result = np.maximum(np.asarray(values, dtype=np.float64), 0)
    reference = frame.session6.to_numpy(dtype=np.float64) if anchor is None else np.asarray(anchor, dtype=np.float64)
    totals = frame[GROUP_KEYS].assign(pred=result, reference=reference).groupby(GROUP_KEYS, sort=False).transform('sum')
    reference_total = totals.reference.to_numpy()
    prediction_total = totals.pred.to_numpy()
    out = np.divide(result * reference_total, prediction_total, out=reference.copy(), where=prediction_total > 0)
    predicted = frame[GROUP_KEYS].assign(value=out).groupby(GROUP_KEYS, sort=False).value.sum()
    expected = frame[GROUP_KEYS].assign(value=reference).groupby(GROUP_KEYS, sort=False).value.sum()
    if not np.isfinite(out).all() or not np.allclose(predicted, expected, rtol=1e-10, atol=1e-5):
        raise ValueError('Не удалось сохранить заданный объём')
    return out

def shape_matrix(frame):
    return np.column_stack([with_volume(frame, frame[name]) for name in BASES])

def fit_weights(x, y):
    y = np.asarray(y, dtype=np.float64)
    denominator = y.sum()
    if denominator <= 0:
        raise ValueError('Нулевая сумма фактов для подбора весов')
    weights = np.zeros(x.shape[1])
    weights[0] = 1
    prediction = x[:, 0].copy()
    best = np.abs(prediction - y).sum() / denominator
    for step in (.2, .05):
        for _ in range(20):
            choice = None
            for source in np.flatnonzero(weights >= step - 1e-10):
                for destination in range(len(weights)):
                    if source == destination:
                        continue
                    amount = min(step, weights[source])
                    trial = prediction + amount * (x[:, destination] - x[:, source])
                    loss = np.abs(trial - y).sum() / denominator
                    if loss < best - 1e-9:
                        best = loss
                        choice = source, destination, amount, trial
            if choice is None:
                break
            source, destination, amount, prediction = choice
            weights[source] -= amount
            weights[destination] += amount
    return weights

def meta_features(frame, matrix, baseline):
    level = np.maximum(frame.level24.to_numpy(dtype=np.float64), 1)
    context = pd.DataFrame(dict(
        log_level=np.log1p(level), current=frame.current.to_numpy() / level,
        delta1h=frame.delta1h.to_numpy() / level, delta2h=frame.delta2h.to_numpy() / level,
        log_network=np.log1p(frame.network_current.to_numpy()),
        horizon=frame.horizon.to_numpy(), dayofweek=frame.cutoff.dt.day_of_week.to_numpy(),
        shape_disagreement=matrix.std(axis=1) / level, blend=baseline / level))
    for index, name in enumerate(BASES):
        context[f'{name}_at_anchor_volume'] = matrix[:, index] / level
        context[f'{name}_minus_session6'] = (matrix[:, index] - matrix[:, 0]) / level
    return context.clip(-50, 50).to_numpy(dtype=np.float32), level

def metric(y, prediction):
    actual = np.asarray(y, dtype=np.float64).ravel()
    predicted = np.asarray(prediction, dtype=np.float64).ravel()
    volume = actual.sum()
    if actual.shape != predicted.shape or not np.isfinite(actual).all() or not np.isfinite(predicted).all():
        raise ValueError('Некорректный вход метрики')
    if volume == 0:
        return dict(rows=len(actual), WAPE=np.nan, relative_bias=np.nan, score=np.nan)
    error = predicted - actual
    wape = np.abs(error).sum() / volume
    bias = error.sum() / volume
    return dict(rows=len(actual), WAPE=wape, relative_bias=bias, score=wape + abs(bias))

def fit_meta(frame):
    matrix = shape_matrix(frame)
    weights = fit_weights(matrix, frame.y)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    model = CatBoostRegressor(depth=4, iterations=220, loss_function='MAE', learning_rate=.035,
                              l2_leaf_reg=20, random_seed=SEED, verbose=False, allow_writing_files=False,
                              task_type='CPU', thread_count=THREADS)
    model.fit(features, (frame.y.to_numpy(dtype=float)-baseline)/level, sample_weight=level/level.mean())
    return model, weights

def predict_meta(frame, fitted):
    model, weights = fitted
    matrix = shape_matrix(frame)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    return with_volume(frame, baseline + np.clip(model.predict(features), -.5, .5)*level)

def score_mix(frame, shape, weight):
    prediction = with_volume(frame, (1-weight)*frame.session6.to_numpy() + weight*shape)
    daily = pd.DataFrame([dict(day=str(day.date()), **metric(group.y, prediction[group.index]))
                          for day, group in frame.groupby('cutoff')])
    summary = dict(**metric(frame.y, prediction), daily_mean=daily.score.mean(),
                   daily_p90=daily.score.quantile(.9), daily_worst=daily.score.max())
    return summary, daily, prediction


In [ ]:
def load_history(path):
    history = pd.read_parquet(path,columns=['timestamp','route_id',*SIGNALS]).sort_values(['timestamp','route_id'])
    times = pd.DatetimeIndex(history.timestamp.unique())
    routes = np.sort(history.route_id.unique())
    expected_times = pd.date_range(times[0],times[-1],freq=STEP)
    if (not times.equals(expected_times) or history.duplicated(['timestamp','route_id']).any()
            or len(history)!=len(times)*len(routes) or len(routes)!=1000):
        raise ValueError('Неполная панель')
    values = history[SIGNALS].to_numpy(dtype=np.float32).reshape(len(times),len(routes),len(SIGNALS))
    if not np.isfinite(values).all() or np.any(values<0):
        raise ValueError('Некорректные значения истории')
    totals = values.sum(axis=1,dtype=np.float64)
    shares = np.divide(values,totals[:,None,:],out=np.zeros_like(values),where=totals[:,None,:]>0)
    return times,routes,totals,shares

def feature_block(origins,h):
    same_time = (shares[origins-48]+shares[origins-336]+shares[origins-672])/3
    current = shares[origins]
    target_slot = origins+h
    prior = (shares[target_slot-48,:,0]+shares[target_slot-336,:,0]+shares[target_slot-672,:,0])/3
    prior = np.maximum(prior,1e-6)
    short = (current+shares[origins-1]+shares[origins-2]+shares[origins-3])/4
    features = [np.log1p(VOLUME_SCALE*current), np.log1p(VOLUME_SCALE*same_time),
                np.log1p(VOLUME_SCALE*short), np.log1p(VOLUME_SCALE*current)-np.log1p(VOLUME_SCALE*same_time),
                np.log1p(VOLUME_SCALE*prior)[...,None]]
    log_current = np.log1p(VOLUME_SCALE*current)
    for lag in (1,2,4):
        features.append(log_current-np.log1p(VOLUME_SCALE*shares[origins-lag]))
    deviation = log_current-np.log1p(VOLUME_SCALE*same_time)
    features.append(deviation[...,1:]-deviation[...,:1])
    seasonal = np.stack([shares[target_slot-lag,:,0] for lag in (48,96,336,672)],axis=0)
    features.append(np.log1p(VOLUME_SCALE*np.std(seasonal,axis=0))[...,None])
    network = np.log1p(network_totals[origins])-np.log1p(network_totals[origins-48])
    features.append(np.broadcast_to(network[:,None,:].astype(np.float32),prior.shape+(network.shape[-1],)))
    shape = prior.shape+(1,)
    features += [np.full(shape,h,dtype=np.float32),
                 np.broadcast_to((minutes[origins]/60).astype(np.float32)[:,None,None],shape),
                 np.broadcast_to(times.day_of_week.to_numpy(dtype=np.float32)[origins,None,None],shape)]
    matrix = np.concatenate(features,axis=2).reshape(len(origins)*len(routes),-1)
    return matrix,prior.reshape(-1)

def make_training():
    blocks = []
    targets = []
    weights = []
    timestamps = []
    for h in range(1,9):
        origins = np.flatnonzero(np.isin(minutes,START_MINUTES)&(np.arange(len(times))>=672)&(np.arange(len(times))+h<len(times)))
        origins = origins[target_volume[origins+h]>0]
        matrix,prior = feature_block(origins,h)
        future = origins+h
        blocks.append(matrix)
        targets.append(shares[future,:,0].reshape(-1)/prior)
        weights.append((prior*np.repeat(target_volume[future],len(routes))).astype(np.float32))
        timestamps.append(np.repeat(times[future].to_numpy(),len(routes)))
    return np.concatenate(blocks),np.concatenate(targets),np.concatenate(weights),np.concatenate(timestamps)

def fit_expert(boundary):
    keep = train_target_time < np.datetime64(pd.Timestamp(boundary))
    weights = train_weights[keep]
    model = CatBoostRegressor(task_type='GPU',devices='0',loss_function='MAE',
                              iterations=500,depth=7,learning_rate=.055,border_count=64,
                              l2_leaf_reg=18,random_seed=42,allow_writing_files=False,
                              thread_count=4,verbose=100)
    print('GPU CatBoost: граница целей',boundary,'строк',int(keep.sum()),flush=True)
    model.fit(train_features[keep],train_target[keep],sample_weight=weights/weights.mean())
    return model

def predict_shares(model,cutoffs):
    cutoffs = pd.DatetimeIndex(cutoffs).unique()
    origins = times.get_indexer(cutoffs)
    if (origins<672).any():
        raise ValueError('Не хватает истории на старте')
    blocks = []
    for h in range(1,9):
        features,prior = feature_block(origins,h)
        share = np.maximum(model.predict(features),0)*prior
        blocks.append(pd.DataFrame(dict(cutoff=cutoffs.repeat(len(routes)),horizon=h,
                                        route_id=np.tile(routes,len(cutoffs)),new_share=share)))
    return pd.concat(blocks,ignore_index=True)

def expert_forecast(frame,model):
    predictions = predict_shares(model,frame.cutoff.unique())
    joined = frame[KEYS].merge(predictions,on=KEYS,validate='one_to_one',how='left')
    if joined.new_share.isna().any():
        raise ValueError('Не совпали ключи прогнозов')
    return with_volume(frame,joined.new_share.to_numpy())

def compare(frame,baseline,expert,period,fold,weights):
    rows = []
    for weight in weights:
        mixed = with_volume(frame,(1-weight)*baseline+weight*expert)
        daily = [metric(f.y,mixed[f.index])['score'] for _,f in frame.groupby('cutoff')]
        rows.append(dict(period=period,fold=fold,weight=weight,**metric(frame.y,mixed),
                         daily_mean=float(np.mean(daily)),daily_p90=float(np.quantile(daily,.9))))
    return rows


In [ ]:
times,routes,network_totals,shares = load_history(DATA/'train_solo_track.parquet')
target_volume = network_totals[:,0]
minutes = (times.hour*60+times.minute).to_numpy()
train_features,train_target,train_weights,train_target_time = make_training()
print('Историческая матрица:',train_features.shape,flush=True)
oof = read_panel(OOF_PATH,'base_oof.parquet','2025-09-20','2025-10-18')
weights = [0.,.10,.25]
rows = []
for start,end in [('2025-10-04','2025-10-11'),('2025-10-11','2025-10-18')]:
    train = oof.loc[oof.cutoff+oof.horizon*STEP < pd.Timestamp(start)].reset_index(drop=True)
    val = oof.loc[(oof.cutoff>=start)&(oof.cutoff<end)].reset_index(drop=True)
    meta = fit_meta(train)
    expert = fit_expert(start)
    for period,frame in [('train_oof',train),('val',val)]:
        rows.extend(compare(frame,predict_meta(frame,meta),expert_forecast(frame,expert),period,start,weights))
    display(pd.DataFrame(rows).loc[lambda table: table.fold.eq(start)])

validation = pd.DataFrame(rows)
pivot = validation.loc[validation.period.eq('val')].pivot(index='fold',columns='weight',values='daily_mean')
eligible = [w for w in weights[1:] if pivot[w].lt(pivot[0.]).all()]
winner = min(eligible,key=lambda w: (pivot[w].mean(),w)) if eligible else 0.
validation.to_csv(OUT/'validation_metrics.csv',index=False)
(OUT/'frozen_selection.json').write_text(json.dumps(dict(weight=winner,criterion='better on both validation weeks',test_used_for_weight_selection=False),indent=2))
print('Замороженный вес нового эксперта:',winner)


In [ ]:
test = read_panel(TEST_PATH,'test_predictions.parquet','2025-10-18','2025-11-01')
meta = fit_meta(oof)
expert = fit_expert('2025-10-18')
for period,frame in [('train_oof',oof),('test',test)]:
    rows.extend(compare(frame,predict_meta(frame,meta),expert_forecast(frame,expert),period,'2025-10-18',list(dict.fromkeys([0.,winner,1.]))))
metrics = pd.DataFrame(rows)
metrics.to_csv(OUT/'metrics.csv',index=False)
display(metrics.loc[metrics.fold.eq('2025-10-18')])
test_metrics = metrics.loc[metrics.period.eq('test')].set_index('weight')
promoted = bool(winner>0 and test_metrics.loc[winner,'daily_mean']<=test_metrics.loc[0.,'daily_mean']-.0003
                and test_metrics.loc[winner,'score']<=test_metrics.loc[0.,'score'])
print('Test оценён до полного переобучения. Train OOF содержит обучающие строки метамодели.')
print('Прошёл порог для сабмита:',promoted)

(OUT/'decision.json').write_text(json.dumps(dict(promoted=bool(promoted),weight=winner,test_before_full_refit=True),indent=2))


In [ ]:
if promoted and PARENT.is_file():
    request = pd.read_parquet(DATA/'test_solo_track.parquet',columns=['id','route_id','timestamp'])
    parent = pd.read_csv(PARENT)
    if (list(parent.columns)!=['id','y_pred'] or len(parent)!=8000 or len(request)!=8000
            or not parent.id.is_unique or not request.id.is_unique or set(parent.id)!=set(request.id)
            or not np.isfinite(parent.y_pred).all() or parent.y_pred.lt(0).any()):
        raise ValueError('Некорректный родительский сабмит')
    cutoff = request.timestamp.min()-STEP
    if times[-1]!=cutoff:
        raise ValueError('Граница истории не совпала с конкурсным стартом')
    final_model = fit_expert(cutoff+STEP)
    query = request.merge(parent,on='id',validate='one_to_one',sort=False)
    query['cutoff'] = cutoff
    query['horizon'] = ((query.timestamp-cutoff)/STEP).astype(int)
    query = query.merge(predict_shares(final_model,[cutoff]),on=KEYS,how='left',validate='one_to_one',sort=False)
    if query.new_share.isna().any():
        raise ValueError('Не совпали конкурсные ключи')
    query['session6'] = query.y_pred
    adjusted = with_volume(query,query.new_share.to_numpy())
    prediction = with_volume(query,(1-winner)*query.y_pred.to_numpy()+winner*adjusted)
    destination = OUT/'submission_pulse_cat_full_history.csv'
    query[['id']].assign(y_pred=prediction).to_csv(destination.with_suffix('.partial'),index=False)
    destination.with_suffix('.partial').replace(destination)
    print('Сабмит:',destination)
elif promoted:
    print('Загрузите родительский сабмит:',PARENT)
else:
    print('Сохраняем текущий конкурсный прогноз. Новый эксперт не прошёл порог.')
print('Результаты:',OUT)
